***
# Generative Artificial Intelligence : Building a Custom Knowledge Base with Amazon Bedrock

***
### [**John Pauline Pineda**](https://github.com/JohnPaulinePineda) <br> <br> *October 10, 2026*
***

* [**1. Table of Contents**](#TOC)
    * [1.1 Project Overview](#1.1)
    * [1.2 Learning Objectives](#1.2)
    * [1.3 Architecture and Service Map](#1.3)
    * [1.4 Configuration Decision Log](#1.4)
    * [1.5 Project Implementation](#1.5)
        * [1.5.1 Step 1](#1.5.1)
        * [1.5.2 Step 2](#1.5.2)
    * [1.6 Validation and Testing](#1.6)
    * [1.7 Project Results](#1.7)
    * [1.8 Metrics and Observability](#1.8)
    * [1.9 Security](#1.9)
    * [1.10 Cost Considerations](#1.10)
    * [1.11 Troubleshooting and Failure Logs](#1.10)
    * [1.12 Challenges and Future Considerations](#1.10)
    * [1.13 Project Cleanup](#1.10)
* [**2. Summary**](#Summary)   
* [**3. References**](#References)

***

# 1. Table of Contents <a class="anchor" id="TOC"></a>

## 1.1. Project Overview <a class="anchor" id="1.1"></a>

This project focuses on building a **Retrieval-Augmented Generation (RAG)** solution using **Amazon Bedrock Knowledge Bases** to enable a foundation model to generate responses grounded in information contained within user-provided documents. Rather than relying exclusively on the knowledge encoded in a foundation model during its original training, the project demonstrates how external organizational knowledge can be incorporated into a generative AI application through a managed knowledge retrieval workflow. The implementation begins with source documents stored in **Amazon Simple Storage Service (Amazon S3)**, which serve as the foundation for the knowledge base. These documents are processed through **document parsing** and **chunking**, where their contents are extracted and divided into smaller segments suitable for retrieval. The resulting text chunks are transformed into numerical representations through an **embedding model**, allowing semantically related pieces of information to be identified based on their meaning rather than solely through exact keyword matching. The generated embeddings are subsequently stored in a **vector store**, with **Amazon OpenSearch Serverless** used in this project to provide a managed environment for vector-based retrieval. Once the knowledge base has been created and the source documents synchronized, the resulting knowledge repository can be queried through the **Amazon Bedrock testing interface** to examine how relevant document passages are retrieved and incorporated into generated responses. The project therefore provides a practical exploration of the complete knowledge-base workflow, from document storage and parsing through chunking, embedding generation, vector storage, retrieval, and grounded response generation. Particular attention is given to the configuration decisions available during knowledge-base creation, including document parsing approaches, chunking strategies, embedding-model selection, vector-store configuration, IAM permissions, and data-source settings, with the objective of understanding not only how each component is configured but also why a particular configuration may be appropriate for a given use case. The project also emphasizes the distinction between a conventional foundation-model interaction and a RAG-based interaction, demonstrating how retrieval of external knowledge can provide a mechanism for grounding model responses in a controlled document collection. Through the combination of AWS Console exploration, configuration analysis, visual documentation, testing, troubleshooting, and evaluation of the resulting retrieval behavior, this project aims to develop practical familiarity with **Amazon Bedrock Knowledge Bases** while establishing a foundation for subsequently incorporating knowledge retrieval into larger generative AI solutions. All implementation steps, configuration decisions, observations, challenges, and relevant references are consolidated throughout this document to provide a reproducible record of the project and a practical reference for future AWS and generative AI development.

[**Retrieval-Augmented Generation (RAG)**](https://www.oreilly.com/library/view/generative-ai-on/9781098159214/) is an approach to generative AI in which information is retrieved from an external knowledge source and provided to a foundation model as contextual information before the model generates a response. Instead of requiring the foundation model to contain all relevant information within its pretrained parameters, RAG separates knowledge retrieval from language generation, allowing applications to ground responses in information contained within documents, databases, or other external sources. A typical RAG workflow consists of two major stages: an ingestion process that prepares source information for retrieval and an inference process that retrieves relevant information and supplies it to the foundation model as context. This architecture is particularly useful when applications need to work with private, domain-specific, frequently changing, or organization-specific information that may not be represented in the model's original training data.

[**Amazon Bedrock Knowledge Bases**](https://aws.amazon.com/bedrock/) is a managed Amazon Bedrock capability that simplifies the creation of knowledge repositories for generative AI applications by providing functionality for ingesting source data, generating embeddings, storing those embeddings, and retrieving relevant information for foundation-model applications. Knowledge Bases can therefore provide the retrieval component of a RAG architecture without requiring the developer to independently implement every stage of the document-ingestion and vector-search pipeline. In this project, Amazon Bedrock Knowledge Bases acts as the central orchestration layer connecting the source documents, parsing and chunking processes, embedding model, vector store, and retrieval workflow. This provides an opportunity to examine how a managed AWS service abstracts portions of the underlying RAG infrastructure while still exposing important configuration decisions to the developer.

[**Amazon Simple Storage Service (Amazon S3)**](https://aws.amazon.com/s3/) provides the source-document storage layer for the knowledge base. In this project, an S3 bucket is used to contain the documents that will become the knowledge base's searchable information, with a Jeff Bezos 2022 shareholder letter used as the source document for the implementation. When the knowledge base is synchronized, Amazon Bedrock accesses the configured S3 location and processes the documents for ingestion. The use of S3 also illustrates an important AWS architectural pattern in which durable object storage serves as the source of data for downstream managed services and processing workflows.

[**Document Parsing**](https://www.oreilly.com/library/view/generative-ai-design/9798341622654/) is the process of extracting usable information from source files before the information is divided into chunks and converted into embeddings. Amazon Bedrock Knowledge Bases provides different parsing approaches depending on the characteristics of the source material, including the **Amazon Bedrock Default Parser**, **Amazon Bedrock Data Automation**, and **Foundation Models as Parsers**, depending on the capabilities available for the selected configuration. The default parser is generally appropriate for conventional text-oriented documents, whereas more advanced parsing approaches can be useful when documents contain complex layouts, tables, images, or other multimodal content. The choice of parser therefore represents an architectural and cost-related consideration rather than simply a technical configuration setting.

[**Chunking**](https://www.oreilly.com/library/view/generative-ai-design/9798341622654/) is the process of dividing extracted document content into smaller segments that can be independently represented, indexed, and retrieved. Chunk size and the amount of contextual overlap influence the granularity of retrieval: excessively large chunks may introduce irrelevant information into the retrieved context, whereas excessively small chunks may remove important contextual relationships. In this project, the default chunking configuration is explored to understand how Amazon Bedrock prepares documents for retrieval. Chunking is an important component of RAG because the quality of the retrieved context directly influences the information subsequently provided to the foundation model.

[**Embeddings**](https://www.oreilly.com/library/view/generative-ai-design/9798341622654/) are numerical representations of information that encode semantic characteristics of text into vectors within a mathematical space. Text with similar meanings can be represented by vectors that are relatively close to one another, allowing systems to perform semantic similarity searches rather than depending exclusively on exact keyword matches. In this project, an Amazon Bedrock-supported embedding model is used to transform document chunks into vector representations before they are stored in the vector database. In practical terms, embeddings transform human-readable knowledge into a mathematical representation that enables an AI system to search for information according to semantic similarity.

[**Vector Stores**](https://www.oreilly.com/library/view/generative-ai-design/9798341622654/) are data stores designed to efficiently index and retrieve high-dimensional vector representations. In a RAG architecture, the vector store provides the retrieval layer that allows a user's query to be compared against the embeddings generated from the source documents so that semantically relevant document chunks can be identified. Amazon Bedrock Knowledge Bases can integrate with several supported storage technologies, including **Amazon OpenSearch Serverless**, **Amazon Aurora PostgreSQL-compatible databases using pgvector**, and other supported retrieval architectures depending on the selected configuration. This project uses **Amazon OpenSearch Serverless** to explore a managed vector-search architecture without requiring the implementation and maintenance of a dedicated vector database infrastructure.

[**Amazon OpenSearch Serverless**](https://aws.amazon.com/opensearch-service/features/serverless/) provides a serverless deployment model for search and analytics workloads and can be used as the vector store underlying an Amazon Bedrock Knowledge Base. Within this project, OpenSearch Serverless stores the vector representations generated from the document chunks and supports the semantic retrieval process used by the knowledge base. Its integration with Bedrock provides an example of how AWS managed services can be combined to construct a RAG architecture in which the developer does not need to manually implement the underlying vector-indexing infrastructure.

[**Knowledge Base Ingestion and Synchronization**](https://www.oreilly.com/library/view/generative-ai-on/9781098159214/) is the process through which the source documents are transformed into searchable knowledge. During synchronization, Amazon Bedrock processes the documents, extracts their content, divides the content into chunks, generates embeddings for those chunks, and stores the resulting representations in the configured vector store. Synchronization therefore represents the transition between the source-document layer and the retrieval-ready knowledge layer. Understanding this process is important because creating a Knowledge Base does not by itself mean that the source documents are immediately available for retrieval; the data source must be ingested and synchronized before meaningful retrieval testing can take place.

[**Semantic Retrieval**](https://www.oreilly.com/library/view/generative-ai-design/9798341622654/) is the process of identifying information that is conceptually relevant to a query based on the similarity of its vector representation to the vectors representing stored document chunks. Unlike traditional keyword search, semantic retrieval can identify relevant information even when the wording of the query differs substantially from the wording contained in the source document. In a RAG workflow, the retrieved chunks provide the contextual evidence that can subsequently be supplied to a foundation model for response generation. The quality of this retrieval stage is therefore a critical factor in determining whether a RAG application produces responses that are relevant and grounded in its source information.

[**Grounded Generation**](https://www.oreilly.com/library/view/generative-ai-design/9798341622654/) refers to the generation of a foundation-model response using retrieved external information as contextual input. In a Knowledge Base-driven RAG workflow, the model is not simply asked to answer a question based on its pretrained knowledge; instead, relevant information retrieved from the configured knowledge source is incorporated into the generation process. This can help applications provide responses that are more closely tied to their designated information sources and can allow users to inspect the retrieved source material supporting an answer. Grounding, however, does not guarantee correctness, making retrieval quality, source-document quality, prompt configuration, and appropriate evaluation important considerations when developing production RAG applications.

[**Amazon Bedrock Knowledge Base Testing**](https://www.oreilly.com/library/view/generative-ai-on/9781098159214/) provides an interactive environment for evaluating the behavior of the configured knowledge base before incorporating it into a larger application. In this project, natural-language questions will be submitted through the testing interface to examine whether relevant document chunks are retrieved and whether the resulting responses are appropriately grounded in the source material. The testing process provides an opportunity to inspect retrieval results, compare questions with different formulations, and identify potential issues in document parsing, chunking, embedding, or retrieval configuration. This serves as an initial validation layer before the knowledge base is integrated into a broader generative AI application.


## 1.2. Learning Objectives <a class="anchor" id="1.2"></a>

1. **Understand the architecture and fundamental components of a typical Retrieval-Augmented Generation (RAG) system independently of Amazon Bedrock.** Develop a conceptual understanding of the major components that constitute a conventional RAG architecture, including source data repositories, document ingestion and preprocessing, document parsing, chunking, embedding models, vector stores, retrieval mechanisms, prompt construction, foundation models, and response generation. Understand the purpose of each component and how these components collectively enable an application to retrieve external knowledge and use it to generate grounded responses.
2. **Explain how the individual components of a typical RAG system operate and interact with one another independently of Amazon Bedrock.** Trace the complete flow of information through a RAG pipeline, from the ingestion of source documents through preprocessing, chunking, embedding generation, vector indexing, query embedding, semantic retrieval, context construction, and final response generation. Understand the interfaces and dependencies between components and how information is transformed as it moves through each stage of the architecture.
3. **Understand the critical design considerations and tradeoffs associated with each component of a RAG system independently of Amazon Bedrock.** Examine how architectural decisions involving document formats, parsing strategies, chunk size, chunk overlap, embedding models, vector stores, similarity-search methods, retrieval parameters, context size, foundation models, latency, scalability, accuracy, security, and cost can affect overall system behavior. Develop an understanding of the tradeoffs between retrieval precision, contextual completeness, computational requirements, response latency, scalability, operational complexity, and cost when designing a RAG system.
4. **Understand how the components of a conventional RAG architecture are implemented, abstracted, or integrated within an Amazon Bedrock Knowledge Base.** Map the conceptual components of a conventional RAG system to their corresponding capabilities within Amazon Bedrock Knowledge Bases. Identify which components are directly configurable, which are managed or abstracted by AWS, and which remain the responsibility of the application developer, thereby understanding how Amazon Bedrock simplifies the construction and operation of a RAG pipeline.
5. **Understand the architecture and operational workflow of Amazon Bedrock Knowledge Bases.** Examine how Amazon Bedrock Knowledge Bases coordinates document ingestion, parsing, chunking, embedding generation, vector storage, synchronization, retrieval, and response generation. Understand the relationship between the Knowledge Base, its data source, embedding model, vector store, retrieval configuration, and foundation model.
6. **Create and configure a data source for an Amazon Bedrock Knowledge Base and understand the document ingestion and synchronization process.** Configure Amazon S3 as a source repository, establish the appropriate data-source location and permissions, and examine how Amazon Bedrock processes documents during synchronization. Understand how ingestion transforms source documents into searchable knowledge and how changes to source data are reflected within the Knowledge Base.
7. **Understand and configure document parsing and chunking strategies within an Amazon Bedrock Knowledge Base.** Explore how different parsing approaches affect the extraction of information from source documents and how chunking determines the units of information subsequently embedded and retrieved. Understand when default configurations may be sufficient and when more specialized parsing or chunking strategies may be appropriate for complex, structured, or multimodal documents.
8. **Configure an embedding model and understand its role in semantic retrieval.** Examine how an embedding model transforms document chunks and user queries into vector representations that enable semantic similarity search. Understand the important considerations when selecting an embedding model, including semantic representation quality, dimensionality, supported data types, model availability, latency, compatibility, and cost, and how embedding choices can influence downstream retrieval performance.
9. **Understand and configure the vector-store layer used by Amazon Bedrock Knowledge Bases.** Examine how Amazon OpenSearch Serverless and other supported vector-storage technologies can provide the retrieval layer for a Knowledge Base. Understand how embeddings are indexed and searched, the role of vector indexes, and the architectural considerations involved in selecting a vector store, including search performance, scalability, integration, operational complexity, availability, and cost.
10. **Query the Knowledge Base using a Retrieval-Augmented Generation workflow and trace the retrieval-to-generation process.** Use natural-language queries to retrieve relevant information from the Knowledge Base and examine how retrieved content is incorporated into foundation-model responses. Trace the flow from the user's query through retrieval and context selection to response generation, with particular attention to how external knowledge influences the resulting answer.
11. **Examine how retrieval configuration affects the relevance, completeness, and grounding of generated responses.** Experiment with retrieval-related configurations and query variations to understand how the number and relevance of retrieved chunks, search behavior, and contextual information can affect generated responses. Investigate the relationship between retrieval quality and generation quality, including situations where inadequate retrieval can lead to incomplete, irrelevant, or poorly grounded responses.
12. **Identify and assess appropriate metrics and qualitative criteria for evaluating RAG system performance.** Develop an understanding of how RAG systems can be evaluated across both retrieval quality and generation quality, including measures and concepts such as retrieval relevance, context precision, context recall, answer relevance, faithfulness/groundedness, correctness, latency, and cost. Recognize that a fluent response is not necessarily a correct or well-grounded response and therefore evaluate the retrieval and generation stages separately where appropriate.
13. **Understand the security, identity, access-control, and data-protection considerations associated with implementing a RAG system within the AWS ecosystem.** Examine the role of AWS Identity and Access Management (IAM), least-privilege permissions, S3 access controls, encryption, service roles, and potentially customer-managed AWS KMS keys in protecting source documents and knowledge-base resources. Identify security considerations associated with sensitive enterprise information and understand how permissions must be established across the services participating in the RAG workflow.
14. **Understand the cost drivers and operational considerations associated with building and running a RAG solution on AWS.** Identify the major potential cost contributors across document storage, document processing, embedding generation, vector storage, retrieval, foundation-model inference, and supporting AWS services. Examine how architectural and configuration decisions can influence cost, and develop practical habits for monitoring resource usage, controlling experimentation costs, and cleaning up resources after completing a project.
15. **Develop a systematic troubleshooting methodology for diagnosing failures across a multi-service RAG architecture.** Learn to distinguish problems originating from source documents, S3 configuration, IAM permissions, document parsing, chunking, embedding generation, vector indexing, retrieval configuration, model invocation, or service availability. Develop a structured approach to examining error messages, configuration settings, service dependencies, logs, and intermediate outputs rather than treating the RAG system as a single black box.
16. **Evaluate the practical limitations of a managed RAG implementation and identify opportunities for future improvement.** Assess what Amazon Bedrock Knowledge Bases abstracts away from the developer and what architectural decisions remain exposed. Identify situations in which a managed Knowledge Base may be appropriate and situations where a more customized RAG architecture might provide greater control over retrieval algorithms, preprocessing, indexing, metadata filtering, evaluation, model selection, orchestration, or application behavior.
17. **Establish a foundation for extending the Knowledge Base into a production-oriented generative AI application.** Use the project as a foundation for subsequently integrating the Knowledge Base with other AWS capabilities, such as Amazon Bedrock Agents, application APIs, serverless compute, monitoring, authentication, and user-facing interfaces. Develop an architectural perspective that moves beyond successfully creating a Knowledge Base toward understanding how the component can participate in a larger, secure, observable, scalable, and cost-conscious generative AI solution.

## 1.3. Architecture and Service Map <a class="anchor" id="1.3"></a>

## 1.4. Configuration Decision Log <a class="anchor" id="1.4"></a>

## 1.5. Project Implementation <a class="anchor" id="1.5"></a>

### 1.5.1. Step 1 <a class="anchor" id="1.5.1"></a>

### 1.5.2. Step 2 <a class="anchor" id="1.5.2"></a>

## 1.6. Validation and Testing <a class="anchor" id="1.6"></a>

## 1.7. Project Results <a class="anchor" id="1.7"></a>

## 1.8. Metrics and Observability <a class="anchor" id="1.8"></a>

## 1.9. Security <a class="anchor" id="1.9"></a>

## 1.10. Cost Considerations <a class="anchor" id="1.10"></a>

## 1.11. Troubleshooting and Failure Logs <a class="anchor" id="1.11"></a>

## 1.12. Challenges and Future Considerations <a class="anchor" id="1.12"></a>

## 1.13. Project Cleanup <a class="anchor" id="1.13"></a>

# 2. Summary <a class="anchor" id="Summary"></a>

# 3. References <a class="anchor" id="References"></a>

* **[Book]** [Generative AI on AWS: Building Context-Aware Multimodal Reasoning Applications](https://www.oreilly.com/library/view/generative-ai-on/9781098159214/) by Chris Fregly, Antje Barth and Shelbee Eigenbrode
* **[Book]** [Generative AI Design Patterns](https://www.oreilly.com/library/view/generative-ai-design/9798341622654/) by Valliappa Lakshmanan and Hannes Hapke
* **[Book]** [Prompt Engineering for Generative AI](https://www.oreilly.com/library/view/prompt-engineering-for/9781098153427/) by James Phoenix and Mike Taylor
* **[Book]** [Introduction to Generative AI](https://www.manning.com/books/introduction-to-generative-ai-second-edition) by Numa Dhamani and Maggie Engler
* **[Book]** [Generative AI in Action](https://www.manning.com/books/generative-ai-in-action) by Amit Bahree
* **[AWS Cloud Service]** [Amazon Bedrock](https://aws.amazon.com/bedrock/) by AWS Team
* **[AWS Cloud Service]** [Amazon S3](https://aws.amazon.com/s3/) by AWS Team
* **[AWS Cloud Service]** [Amazon OpenSearch Serverless](https://aws.amazon.com/opensearch-service/features/serverless/) by AWS Team
* **[Article]** [Introducing Amazon Bedrock Managed Knowledge Base for faster, more accurate enterprise AI applications](https://aws.amazon.com/blogs/aws/introducing-amazon-bedrock-managed-knowledge-base-for-faster-more-accurate-enterprise-ai-applications/) by Daniel Abib
* **[Article]** [Create a knowledge base by connecting to a data source in Amazon Bedrock Knowledge Bases](https://docs.aws.amazon.com/bedrock/latest/userguide/knowledge-base-create.html) by AWS Team
* **[Article]** [Customizing your knowledge base](https://docs.aws.amazon.com/bedrock/latest/userguide/kb-how-customization.html) by AWS Team
* **[Article]** [Create a managed knowledge base](https://docs.aws.amazon.com/bedrock/latest/userguide/kb-managed-create.html) by AWS Team
* **[Article]** [Creating knowledge bases from Amazon Bedrock managed knowledge bases](https://docs.aws.amazon.com/quick/latest/userguide/byo-bedrock-kb-creating.html) by AWS Team
* **[Article]** [Create a Knowledge Base in Amazon Bedrock (Step-by-Step Console Guide)](https://dev.to/dipayan_das/create-a-knowledge-base-in-amazon-bedrock-step-by-step-console-guide-3825) by Dipayan Das
* **[Course]** [Building a Custom Knowledge Base with Amazon Bedrock](https://www.whizlabs.com/labs/building-a-custom-knowledge-base-with-amazon-bedrock/) by Whizlabs Team